In [1]:
! pip install boto3 huggingface_hub minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 649.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 2.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 419.6 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 349.3 kB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 309.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 337.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 208.1 kB/s eta 0:00:00a 0:00:01


In [ ]:
import boto3
import os
from minio import Minio
from huggingface_hub import list_repo_files
import requests

In [8]:
# Configuration
DATASET_ID = "Zihan1004/FNSPID"
S3_BUCKET = "fnspid-bucket"

# MinIO Configuration
MINIO_ENDPOINT = "minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"


In [9]:
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    use_ssl=False
)

In [10]:
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"Bucket '{S3_BUCKET}' already exists")
except:
    s3_client.create_bucket(Bucket=S3_BUCKET)
    print(f"Created bucket '{S3_BUCKET}'")

Bucket 'fnspid-bucket' already exists


In [12]:
print("\nFetching file list from Hugging Face...")
files = list_repo_files(
    repo_id=DATASET_ID,
    repo_type="dataset"
)


Fetching file list from Hugging Face...


In [13]:
files

['README.md',
 'Stock_news/All_external.csv',
 'Stock_news/nasdaq_exteral_data.csv',
 'Stock_price/full_history.zip']

In [17]:
def get_news_data():
    hf_url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/Stock_news/nasdaq_exteral_data.csv"
    response = requests.get(hf_url, stream=True)
    response.raise_for_status()
    s3_key = "raw/stock_news/nasdaq_exteral_data.csv"
    try:
        s3_client.upload_fileobj(
            response.raw,
            S3_BUCKET,
            s3_key
        )
        print("Upload successful!")
    except Exception as e:
        print(f"Error uploading to S3: {e}")

In [18]:
# get_news_data()

In [19]:
def get_stocks_data():
    hf_url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/Stock_price/full_history.zip"
    response = requests.get(hf_url, stream=True)
    response.raise_for_status()
    s3_key = "raw/stock_price/full_history.zip"
    try:
        s3_client.upload_fileobj(
            response.raw,
            S3_BUCKET,
            s3_key
        )
        print("Upload successful!")
    except Exception as e:
        print(f"Error uploading to S3: {e}")

In [20]:
# get_stocks_data()

In [36]:
# Extract stocks price from zip
from io import BytesIO
import zipfile
from tqdm import tqdm

def extract_zip():
    # Get the object and read from the Body
    response = s3_client.get_object(
        Bucket="fnspid-bucket",
        Key="raw/stock_price/full_history.zip"
    )
    zip_bytes = BytesIO(response['Body'].read())

    with zipfile.ZipFile(zip_bytes, "r") as z:
        file_list = [name for name in z.namelist() if "__MACOSX" not in name]

        for name in tqdm(file_list, desc="Extracting ZIP", unit="file"):
            file_data = z.read(name)
            s3_client.put_object(
                Bucket="fnspid-bucket",
                Key=f"raw/stock_price/{name}",
                Body=BytesIO(file_data),
                ContentLength=len(file_data)
            )


In [ ]:
extract_zip()

Extracting ZIP:  34%|███▍      | 2610/7694 [11:15<29:52,  2.84file/s]  